# Government of Canada - Contracts Over $10,000: Exploration & Analysis

**Dataset**: Proactive disclosure of federal contracts, published by the Government of Canada.  
**Source**: [Open Canada Portal](https://open.canada.ca/data/en/dataset/d8f85d91-7dec-4fd1-8055-483b77225d8b)

**Goal**: Explore this dataset to find the most important and actionable insights about federal procurement.

**Approach**:
1. Load and profile the dataset to understand its structure
2. Audit data quality and validate against the data dictionary
3. Clean and prepare the data for analysis
4. Analyze three key questions about competition, cost control, and vendor concentration
5. Export summary tables for dashboard visualization

## 1. Setup & Data Loading

Load the CSV into an in-memory DuckDB table. All columns are loaded as `VARCHAR` (strings) intentionally, this dataset contains code fields like `award_criteria` (values 0–5) and `solicitation_procedure` (values like TN, TC) that would be misinterpreted as numbers or abbreviations if auto-typed. We'll cast explicitly when needed.

In [80]:
import os
import duckdb
import polars as pl
from dotenv import load_dotenv

load_dotenv(override=True)

# Variable Setup
dataset_path = os.getenv("LOCAL_DATASET_PATH")
if not dataset_path:
    raise ValueError('Missing LOCAL_DATASET_PATH. Copy ".env.example" to ".env" and set the path.')

# Create in-memory DuckDB connection
con = duckdb.connect(database=":memory:")

In [81]:
con.execute("""
    CREATE TEMP TABLE contracts AS
    SELECT *
    FROM read_csv_auto(
        '{0}',
        delim = ',',
        header = true,
        strict_mode = false,
        all_varchar = true
    )
""".format(dataset_path))

## 2. Dataset Overview

Let's understand the basic shape of this dataset, that is, how many rows, columns, what time period it covers, and what each row represents.

In [85]:
total_rows = con.sql("SELECT COUNT(*) FROM contracts").fetchone()[0]  # type: ignore
total_cols = len(con.sql("SELECT * FROM contracts LIMIT 1").columns)
distinct_refs = con.sql("SELECT COUNT(DISTINCT reference_number) FROM contracts").fetchone()[0]  # type: ignore
distinct_orgs = con.sql("SELECT COUNT(DISTINCT owner_org) FROM contracts").fetchone()[0]  # type: ignore
distinct_vendors = con.sql("SELECT COUNT(DISTINCT vendor_name) FROM contracts").fetchone()[0]  # type: ignore
date_range = con.sql("""
    SELECT MIN(TRY_CAST(contract_date AS DATE)), MAX(TRY_CAST(contract_date AS DATE))
    FROM contracts
    WHERE TRY_CAST(contract_date AS DATE) BETWEEN '2004-01-01' AND '2025-12-31'
""").fetchone()  # type: ignore

print(f"Rows:                {total_rows:,}")
print(f"Columns:             {total_cols}")
print(f"Unique references:   {distinct_refs:,}")
print(f"Departments:         {distinct_orgs}")
print(f"Distinct vendors:    {distinct_vendors:,}")
print(f"Date range:          {date_range[0]} to {date_range[1]}") # type: ignore

Rows:                1,261,327
Columns:             43
Unique references:   499,862
Departments:         99
Distinct vendors:    208,152
Date range:          2004-01-04 to 2025-12-31


### What is a row?

Each row is **not** a unique contract. It's a reported transaction. The `instrument_type` field tells us what kind:
- `C` = Original contract (or call-up against a standing offer)
- `A` = Amendment to an existing contract
- `SOSA` = Standing Offer / Supply Arrangement (a framework agreement, not a purchase)

A single contract can appear multiple times, that is, once for the original award, then once per amendment. The `reference_number` groups these together.

In [83]:
# Instrument type breakdown
con.sql("""
    SELECT 
        COALESCE(instrument_type, 'null (pre-2019)') AS instrument_type,
        COUNT(*) AS row_count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 1) AS pct
    FROM contracts
    GROUP BY instrument_type
    ORDER BY row_count DESC
""").pl()

instrument_type,row_count,pct
str,i64,f64
"""C""",643474,51.0
"""null (pre-2019)""",354581,28.1
"""A""",191746,15.2
"""SOSA""",71526,5.7


In [84]:
# How many rows share a reference_number? (1 = standalone, 2+ = contract with amendments)
con.sql("""
    SELECT 
        rows_per_ref,
        COUNT(*) AS num_references
    FROM (
        SELECT reference_number, COUNT(*) AS rows_per_ref
        FROM contracts
        GROUP BY reference_number
    )
    GROUP BY rows_per_ref
    ORDER BY rows_per_ref
    LIMIT 10
""").pl()

rows_per_ref,num_references
i64,i64
1,356714
2,43955
3,26061
4,12377
5,11322
6,8188
7,5507
8,4847
9,3815


**Key takeaway**: 357K references appear only once (standalone contracts), but ~143K references appear 2+ times, meaning they have amendments. Those ~143K references account for ~905K rows. This means **we cannot simply sum `contract_value` across all rows** without double-counting amended contracts.

### How do the three money columns relate?

Per the data dictionary:
- `original_value` = what the contract was worth when first signed
- `amendment_value` = the dollar change of a specific amendment (positive or negative)
- `contract_value` = the **cumulative running total** (original + all amendments to date)

On an amendment row, `contract_value` is the *updated total*, not an additional amount.